In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
df_raw=pd.read_csv("discrete_tumor_id3_practice.csv")

In [58]:
df=df_raw.copy()
x_cols=df.drop('diagnosis',axis=1).columns.to_list()
y_col='diagnosis'

#计算信息熵
def caculate_entropy(series:pd.Series)->float:
    if len(series)==0:
        return 0
    prob=series.value_counts(normalize=True)
    return float(-np.sum(prob*np.log2(prob)))

def caculate_conditional_entropy(df:pd.DataFrame,x_col:str,y_col:str)->float:
    prob_xi=df[x_col].value_counts(normalize=True)
    xi_entropy=df.groupby(x_col)[y_col].apply(caculate_entropy)
    return float(np.sum(prob_xi*xi_entropy))


class TreeNode:
    def __init__(self,is_leaf=True,feature=None,prediction=None):
        self.is_leaf=is_leaf
        self.feature=feature
        self.prediction=prediction
        self.children={}
    def display(self, indent="", is_last=True):
        # 1. 决定当前节点的连线符号
        marker = "└── " if is_last else "├── "

        if self.is_leaf:
            print(f"{indent}{marker}🍃 预测结果: [ {self.prediction} ]")
            return

        # 2. 打印当前分裂特征
        print(f"{indent}{marker}🌲 划分特征: [ {self.feature} ]")

        # 3. 计算下一层的缩进
        # 如果当前是最后一个分支，后面就用空格；否则用垂直线 | 延伸下去
        next_indent = indent + ("    " if is_last else "│   ")

        # 4. 递归打印子节点
        child_items = list(self.children.items())
        for i, (key, val) in enumerate(child_items):
            is_last_child = i == len(child_items) - 1
            print(f"{next_indent}├── 当取值为 -> '{key}':")
            # 传给子节点时，告诉它它是不是最后一个子节点
            val.display(indent=next_indent + "│   ", is_last=is_last_child)
    def predict(self,sample:dict):
        current_node=self
        while not current_node.is_leaf:
            feat=current_node.feature
            val=sample.get(feat)
            if val not in current_node.children:
                return "error:unknown feature"
            current_node=current_node.children[val]
        return current_node.prediction
    

def build_tree(df:pd.DataFrame,
               features:list,
               target_col:str,
              entropy_threshold: float = 0.1)->TreeNode:
    current_entropy=caculate_entropy(df[target_col])
    unique_targets=df[target_col].unique()
    if len(unique_targets)==1:
        return TreeNode(is_leaf=True,prediction=unique_targets[0])
    if len(features)==0:
        most_common=df[target_col].mode()[0]
        return TreeNode(is_leaf=True,prediction=most_common)
    if current_entropy<=entropy_threshold:
        most_common=df[target_col].mode()[0]
        return TreeNode(is_leaf=True,prediction=most_common)

    best_feature=None
    best_gain=-1

    for feat in features:
        gain=current_entropy-caculate_conditional_entropy(df,feat,target_col)
        if gain>best_gain:
            best_gain=gain
            best_feature=feat
    if best_gain<1e-5:
        most_common=df[target_col].mode()[0]
        return TreeNode(is_leaf=True,prediction=most_common)
        
    root=TreeNode(is_leaf=False,feature=best_feature)
    remaining_feature=[f for f in features if f !=best_feature]
    
    best_feature_values=df[best_feature].unique()
    for val in best_feature_values:
        sub_df=df[df[best_feature]==val]
        children_node=build_tree(
            sub_df,
            remaining_feature,
            target_col,
            entropy_threshold
        )
        root.children[val]=children_node
    return root
        


    




In [65]:

decision_tree=build_tree(
    df,
    x_cols,
    y_col,
    0.1
)
decision_tree.display()

df["prediction"]=df.apply(
    lambda row:decision_tree.predict(row.to_dict()),axis=1
)
df_test=pd.read_csv(r"D:\UserFiles\Downloads\discrete_tumor_test_answer_key.csv")
df_test["prediction"]=df_test.apply(
    lambda row:decision_tree.predict(row.to_dict()),axis=1
)
acc=np.mean(df_test['diagnosis']==df_test['prediction'])
acc

└── 🌲 划分特征: [ texture ]
    ├── 当取值为 -> 'soft':
    │   ├── 🍃 预测结果: [ benign ]
    ├── 当取值为 -> 'firm':
    │   ├── 🌲 划分特征: [ shape ]
    │   │   ├── 当取值为 -> 'lobulated':
    │   │   │   ├── 🍃 预测结果: [ benign ]
    │   │   ├── 当取值为 -> 'round':
    │   │   │   ├── 🍃 预测结果: [ benign ]
    │   │   ├── 当取值为 -> 'irregular':
    │   │   │   └── 🌲 划分特征: [ age_group ]
    │   │   │       ├── 当取值为 -> 'old':
    │   │   │       │   ├── 🍃 预测结果: [ malignant ]
    │   │   │       ├── 当取值为 -> 'young':
    │   │   │       │   ├── 🍃 预测结果: [ malignant ]
    │   │   │       ├── 当取值为 -> 'middle':
    │   │   │       │   └── 🍃 预测结果: [ benign ]
    ├── 当取值为 -> 'hard':
    │   └── 🍃 预测结果: [ malignant ]


1.0